# DuckDB fix-plan store example

This notebook loads the shared integration-test fix-plan documents into a temporary DuckDB store, then queries the store for plans matching specific synthetic dataset IDs and source paths.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from woodpecker.stores import DuckDBFixPlanStore, JsonFixPlanStore
from woodpecker.testing import (
    integration_root_dir,
    make_atlas,
    make_cmip6,
    make_cmip6_decadal,
    make_cmip7,
)

Load every JSON/YAML plan document from `tests/integration/plans/` into a temporary DuckDB database.

In [ ]:
temporary_directory = TemporaryDirectory()
database_path = Path(temporary_directory.name) / "fix-plans.duckdb"
store = DuckDBFixPlanStore(database_path)

plan_dir = integration_root_dir() / "plans"
plan_paths = sorted(
    path for path in plan_dir.glob("*") if path.suffix.lower() in {".json", ".yaml", ".yml"}
)

for plan_path in plan_paths:
    for plan in JsonFixPlanStore(plan_path).list_plans():
        store.save_plan(plan)

database_path, [plan.id for plan in store.list_plans()]

Create representative datasets and ask the store which plans match each dataset. The lookup uses dataset metadata plus the source path text, so the result is driven by the match rules declared inside each fix plan.

In [ ]:
query_cases = [
    (
        "CMIP6 core",
        make_cmip6(overrides={"units": "degC"}, seed=7),
        "CMIP6.CMIP.MOHC.HadGEM3-GC31-LL.historical.r1i1p1f3.Amon.tas.gn.v20200101.nc",
    ),
    (
        "CMIP6 decadal",
        make_cmip6_decadal(seed=7),
        "CMIP6.DCPP.MPI-M.MPI-ESM1-2-HR.dcppA-hindcast.s1960-r1i1p1f1.Omon.tos.gn.v20200101.nc",
    ),
    (
        "Atlas",
        make_atlas(seed=7),
        "c3s-ipcc-atlas.cmip6.historical.ssp245.pr.monthly.global.nc",
    ),
    (
        "ESA CCI water vapour",
        make_cmip7(variable="prw", seed=7),
        "ESACCI-WATERVAPOUR-L3C-TCWV-meris-005deg-2002-2017-fv3.2.zarr",
    ),
]

In [ ]:
matches = []
for label, dataset, source_path in query_cases:
    matched_plans = store.lookup(dataset, path=source_path)
    matches.append(
        {
            "case": label,
            "dataset_id": dataset.attrs.get("dataset_id", ""),
            "source_path": source_path,
            "plan_ids": [plan.id for plan in matched_plans],
        }
    )

matches

Inspect the selected plan to see the ordered fix IDs that would be used for a dataset.

In [ ]:
selected = store.get_plan("cmip6_decadal.full")

{
    "id": selected.id,
    "description": selected.description,
    "match": selected.match.model_dump() if selected.match else None,
    "steps": [step.id for step in selected.steps],
}

In [ ]:
temporary_directory.cleanup()